# Extended Lab: Regularization for a Mincer Wage Equation

**Dataset:** `wage_survey.csv` — 2,000 simulated workers, 19 candidate predictors, target `log_hourly_wage` (and its level, `hourly_wage`).

**Economic framing:** Labor economists model wages with a **Mincer earnings equation**: `log(wage) = β0 + β1·education + β2·experience + β3·experience² + ... + ε`. Log-wages are used (rather than raw wages) because wage distributions are right-skewed and because coefficients on a log-linear model have a convenient interpretation: `β1` is approximately "the percentage change in wages per additional year of education."

In practice, researchers rarely stop at the textbook four-variable Mincer equation. They add demographic controls, job characteristics, and regional dummies — and once you're including a dozen-plus correlated controls, you're back in exactly the overfitting/multicollinearity territory that motivates regularization in any domain.

## Learning objectives
1. Fit a saturated (many-control) wage regression and diagnose overfitting via train/test MSE and R².
2. Explain why collinear regressors (age, experience, education are mechanically related) destabilize OLS coefficients.
3. Apply Ridge, Lasso, and Elastic Net regression and tune `alpha` with `RidgeCV` / `LassoCV` / `GridSearchCV`.
4. Use Lasso's coefficient-zeroing behavior as a **variable selection** tool and connect it to how applied economists actually use Lasso (e.g., double-selection Lasso for causal inference).
5. Understand *why* regularization's bias is a feature for prediction tasks but a serious caveat for causal/policy questions — a distinction that matters much more in economics than in most ML applications.

## Part 0 — Setup & data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (9, 5)

df = pd.read_csv('wage_survey.csv')
print(df.shape)
df.head()

**Checkpoint 0.1:** We'll predict `log_hourly_wage` (the standard econometric target) and drop the level `hourly_wage` from the feature set to avoid trivially leaking the target.

In [ ]:
y = df['log_hourly_wage']
features = df.drop(columns=['log_hourly_wage', 'hourly_wage'])
predictors = features.columns
print(list(predictors))
df[['hourly_wage', 'log_hourly_wage']].describe()

## Part 1 — EDA: correlation and collinearity

In [ ]:
corr = features.assign(log_hourly_wage=y).corr()
plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False)
plt.title('Feature correlation matrix')
plt.tight_layout()
plt.show()

corr['log_hourly_wage'].sort_values(ascending=False)

**Checkpoint 1.1:** `age`, `experience_years`, and `education_years` are mechanically linked (`age ≈ education + experience + 6`), and `experience_sq` is a deterministic function of `experience_years`. Confirm this and flag it as multicollinearity risk before fitting anything.

In [ ]:
print(features[['age','education_years','experience_years']].corr())
print()
print('corr(experience, experience_sq):', np.corrcoef(features['experience_years'], features['experience_sq'])[0,1])

## Part 2 — Preprocessing: scaling

As in any regularized regression, all features must be on comparable scales before the penalty is applied — otherwise `experience_sq` (values up to ~2000) would be penalized very differently from `union_member` (0/1) purely due to units, not economic importance.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(features)
X = scaler.transform(features)

## Part 3 — Train/test split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(X_train.shape, X_test.shape)

## Part 4 — Baseline: unregularized OLS on the full (saturated) specification

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

ols = LinearRegression()
ols.fit(X_train, y_train)

mse_train = mean_squared_error(y_train, ols.predict(X_train))
mse_test = mean_squared_error(y_test, ols.predict(X_test))
print('Training MSE:', round(mse_train, 4), '| R2:', round(r2_score(y_train, ols.predict(X_train)), 3))
print('Testing  MSE:', round(mse_test, 4), '| R2:', round(r2_score(y_test, ols.predict(X_test)), 3))

In [ ]:
coef = pd.Series(ols.coef_, predictors).sort_values()
coef.plot(kind='bar', title='OLS coefficients (standardized, full specification)')
plt.axhline(0, color='k', linewidth=0.8)
plt.tight_layout()
plt.show()

**Discussion:** with 2,000 observations and only 19 features, OLS won't overfit as dramatically as it would with, say, county-level growth regressions where you might have 40 candidate regressors and 90 countries (the "Sala-i-Martin problem," discussed below) — but you should still see the collinear age/education/experience trio producing coefficients that look less stable or intuitive than a clean four-variable Mincer equation would, and several genuinely irrelevant controls (`sibling_count`, `commute_minutes`, `coffee_cups_per_day`, `household_size`, `savings_rate_pct`) receiving small but nonzero coefficients purely from sampling noise.

## Part 5 — Ridge regression (L2)

In [ ]:
from sklearn.linear_model import Ridge

ridge_default = Ridge(alpha=1.0)
ridge_default.fit(X_train, y_train)
print('Ridge (alpha=1) test MSE:', mean_squared_error(y_test, ridge_default.predict(X_test)))
print('Ridge (alpha=1) test R2 :', r2_score(y_test, ridge_default.predict(X_test)))

## Part 6 — Tuning alpha with RidgeCV and GridSearchCV

In [ ]:
from sklearn.linear_model import RidgeCV

alphas = np.logspace(-3, 4, 100)
ridge_cv = RidgeCV(alphas=alphas, cv=5)
ridge_cv.fit(X_train, y_train)
print('Best alpha:', ridge_cv.alpha_)
print('Test MSE at best alpha:', mean_squared_error(y_test, ridge_cv.predict(X_test)))
print('Test R2  at best alpha:', r2_score(y_test, ridge_cv.predict(X_test)))

In [ ]:
train_mse, test_mse = [], []
for a in alphas:
    r = Ridge(alpha=a).fit(X_train, y_train)
    train_mse.append(mean_squared_error(y_train, r.predict(X_train)))
    test_mse.append(mean_squared_error(y_test, r.predict(X_test)))

plt.plot(alphas, train_mse, label='Training MSE')
plt.plot(alphas, test_mse, label='Test MSE')
plt.axvline(ridge_cv.alpha_, color='k', linestyle='--', label='Best alpha (CV)')
plt.xscale('log')
plt.xlabel('alpha')
plt.ylabel('MSE')
plt.legend()
plt.title('Ridge: bias-variance tradeoff over alpha')
plt.show()

## Part 7 — Lasso regression (L1) and variable selection

In [ ]:
from sklearn.linear_model import LassoCV

lasso_cv = LassoCV(alphas=np.logspace(-4, 1, 100), cv=5, max_iter=10000)
lasso_cv.fit(X_train, y_train)
print('Best alpha:', lasso_cv.alpha_)
print('Test MSE:', mean_squared_error(y_test, lasso_cv.predict(X_test)))
print('Test R2 :', r2_score(y_test, lasso_cv.predict(X_test)))

In [ ]:
coef_lasso = pd.Series(lasso_cv.coef_, predictors).sort_values()
coef_lasso.plot(kind='bar', title='Lasso coefficients (tuned alpha)')
plt.axhline(0, color='k', linewidth=0.8)
plt.tight_layout()
plt.show()

kept = coef_lasso[coef_lasso != 0]
dropped = coef_lasso[coef_lasso == 0]
print(f'Lasso kept {len(kept)} of {len(coef_lasso)} features.')
print('Dropped:', list(dropped.index))

**Checkpoint 7.1:** Compare the dropped list to the "genuinely irrelevant" controls we built into the data-generating process (`sibling_count`, `commute_minutes`, `household_size`, `coffee_cups_per_day`, `savings_rate_pct`) and to the true zero-effect regional/health noise. Lasso should recover most of these correctly — this is a nice illustration of why applied economists reach for Lasso specifically as an automated, principled way to trim a bloated control set rather than hand-picking controls (a practice known to invite specification-search bias).

## Part 8 — Elastic Net

In [ ]:
from sklearn.linear_model import ElasticNetCV

enet_cv = ElasticNetCV(
    alphas=np.logspace(-4, 1, 50), l1_ratio=[.1, .3, .5, .7, .9, .95, .99],
    cv=5, max_iter=10000
)
enet_cv.fit(X_train, y_train)
print('Best alpha:', enet_cv.alpha_, '| best l1_ratio:', enet_cv.l1_ratio_)
print('Test MSE:', mean_squared_error(y_test, enet_cv.predict(X_test)))
print('Test R2 :', r2_score(y_test, enet_cv.predict(X_test)))

## Part 9 — Model comparison

In [ ]:
models = {
    'OLS (no regularization)': ols,
    'Ridge (alpha=1)': ridge_default,
    'Ridge (tuned)': ridge_cv,
    'Lasso (tuned)': lasso_cv,
    'Elastic Net (tuned)': enet_cv,
}

rows = []
for name, m in models.items():
    pred_test = m.predict(X_test)
    rows.append({
        'Model': name,
        'Test MSE': mean_squared_error(y_test, pred_test),
        'Test R2': r2_score(y_test, pred_test),
        'Nonzero coefficients': int(np.sum(np.abs(m.coef_) > 1e-8)) if hasattr(m, 'coef_') else len(predictors),
    })

pd.DataFrame(rows).set_index('Model').round(4)

## Part 10 — Converting a standardized coefficient back to an economically meaningful statement

Because we standardized `X`, coefficients are in "per standard deviation" units, not "per year of education." Economists usually want the latter. Below we refit **unstandardized** Ridge/Lasso at their tuned alphas (scaled proportionally, since alpha's effective strength depends on the feature scale) just to read off an approximate percentage return to education — purely for interpretability, not for the tuning exercise itself.

In [ ]:
# Quick illustrative unstandardized OLS for a clean textbook-style interpretation
ols_raw = LinearRegression().fit(features, y)
edu_coef = ols_raw.coef_[list(predictors).index('education_years')]
print(f"Unstandardized OLS: one more year of education is associated with "
      f"approximately {edu_coef*100:.1f}% higher hourly wages, holding other controls fixed.")

## Part 11 — Why this matters differently in economics: prediction vs. causal inference

Everything above optimizes **predictive accuracy** (minimizing test MSE). But a labor economist estimating the return to education usually wants an **unbiased causal estimate** of `β_education`, not just a model that predicts wages well. This is where regularization's core mechanism — deliberately introducing bias to reduce variance — becomes a genuine tension rather than a free lunch:

- Ridge and Lasso coefficients are **biased toward zero by construction**. That's fine if your goal is prediction. It is a direct problem if you want to report "the" effect of education on wages, because the reported coefficient is *systematically* attenuated.
- The applied econometrics literature (Belloni, Chernozhukov, and Hansen, among others) developed **double/debiased Lasso** specifically to fix this: use Lasso only to *select* which control variables belong in a low-dimensional final regression (partialing out confounders via the Frisch–Waugl–Lovell theorem), then run plain, unbiased OLS on the selected specification for the coefficient of interest. Lasso is used as a *selection device*, not as the estimator whose coefficients you report.
- **The "two million regressions" problem** (Sala-i-Martin, 1997) in cross-country growth economics is the classic illustration of why economists needed a principled way to handle many candidate controls before methods like Lasso were common — researchers were running enormous numbers of regressions with different control-variable subsets and reporting whichever looked good, a form of specification-search / p-hacking that automated, penalized variable selection helps guard against.

**Checkpoint 11.1 (reflection):** If your goal were to report a policy-relevant estimate of the union wage premium (the `union_member` coefficient), would you report the Lasso-tuned coefficient directly, or would you use Lasso only to select controls and then re-estimate with OLS? Justify your answer using the reasoning above.